# Lesson 10: UniversalNER (UniNER) - LLM-Distilled Open-Domain NER

## Overview

**UniversalNER** (also known as **UniNER**) represents a breakthrough in open-domain Named Entity Recognition by distilling the capabilities of large language models (LLMs) like ChatGPT into smaller, efficient models that can recognize virtually any entity type without training.

### What Makes UniversalNER Special?

| Feature | Traditional NER | UniversalNER |
|---------|-----------------|---------------|
| Entity Types | Fixed (PER, ORG, LOC) | Any type specified at inference |
| Training | Requires labeled data | Pre-trained on diverse annotations |
| Model Size | 110M-340M parameters | 7B-13B parameters |
| Domain Adaptation | Needs fine-tuning | Zero-shot transfer |
| Annotation Cost | High | Zero (uses LLM distillation) |

### Key Innovation: Targeted Distillation

UniversalNER was created by:
1. **Prompting ChatGPT** to generate diverse entity annotations across many domains
2. **Creating a massive dataset** of 45,889 entity types from the Pile corpus
3. **Distilling this knowledge** into a Llama model using instruction tuning

### Learning Objectives

By the end of this lesson, you will:
- Understand the UniNER architecture and training methodology
- Use UniNER for zero-shot NER on any entity type
- Compare UniNER with GLiNER and traditional approaches
- Implement efficient inference pipelines
- Understand the strengths and limitations of LLM-distilled NER

### References

- **Paper**: [UniversalNER: Targeted Distillation from Large Language Models for Open Named Entity Recognition](https://arxiv.org/abs/2308.03279) (Zhou et al., 2023)
- **GitHub**: [https://github.com/universal-ner/universal-ner](https://github.com/universal-ner/universal-ner)
- **Model Hub**: [https://huggingface.co/Universal-NER](https://huggingface.co/Universal-NER)

## 1. Environment Setup

UniversalNER requires significant GPU memory due to its 7B parameter size. We'll use 4-bit quantization to make it runnable on consumer GPUs.

In [ ]:
# Install required packages
!pip install -q transformers>=4.35.0 accelerate>=0.24.0 bitsandbytes>=0.41.0 sentencepiece protobuf

# For comparison with other methods
!pip install -q gliner seqeval

print("✓ Installation complete!")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import re
import json
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Understanding the UniversalNER Architecture

### 2.1 The Targeted Distillation Process

UniversalNER's training follows a unique "targeted distillation" paradigm:

```
┌─────────────────────────────────────────────────────────────────┐
│                    TARGETED DISTILLATION                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Step 1: Entity Type Generation                                 │
│  ┌────────────────┐     Prompt: "What entity types              │
│  │   ChatGPT      │───→ exist in this passage?"                 │
│  └────────────────┘                                             │
│         │                                                       │
│         ▼                                                       │
│  Step 2: Entity Extraction                                      │
│  ┌────────────────┐     Prompt: "Extract all {type}             │
│  │   ChatGPT      │───→ entities from this text"                │
│  └────────────────┘                                             │
│         │                                                       │
│         ▼                                                       │
│  Step 3: Dataset Creation                                       │
│  ┌────────────────────────────────────────────────────────┐    │
│  │ 45,889 entity types, millions of examples from Pile    │    │
│  └────────────────────────────────────────────────────────┘    │
│         │                                                       │
│         ▼                                                       │
│  Step 4: Instruction Tuning                                     │
│  ┌────────────────┐     Fine-tune on NER instruction            │
│  │  Llama-7B/13B  │───→ format for open-domain NER              │
│  └────────────────┘                                             │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 2.2 Instruction Format

UniNER uses a specific conversation format:

```
A virtual assistant answers questions from a user based on the provided text.
User: Text: {input_text}
Assistant: I've read this text.
User: What describes {entity_type} in the text?
Assistant: {extracted_entities}
```

### 2.3 Model Variants

| Model | Base | Parameters | Performance |
|-------|------|------------|-------------|
| UniNER-7B-type | Llama-7B | 7B | Type-supervised |
| UniNER-7B-all | Llama-7B | 7B | All data (best zero-shot) |
| UniNER-7B-sup | Llama-7B | 7B | Supervised only |

## 3. Loading UniversalNER with Quantization

We'll use 4-bit quantization to reduce memory requirements from ~14GB to ~4GB.

In [ ]:
# Configure 4-bit quantization for efficient inference
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load UniNER-7B-all (best zero-shot performance)
MODEL_NAME = "Universal-NER/UniNER-7B-all"

print(f"Loading {MODEL_NAME}...")
print("This may take a few minutes on first run due to model download (~14GB)")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="left"  # Important for batch generation
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print(f"\n✓ Model loaded successfully!")
print(f"Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 4. Building the UniNER Extraction Pipeline

### 4.1 The Prompt Template

UniNER requires a specific prompt format that mimics a conversation.

In [ ]:
class UniNERExtractor:
    """A clean wrapper for UniversalNER entity extraction."""
    
    # The official prompt template from the UniNER paper
    PROMPT_TEMPLATE = """A virtual assistant answers questions from a user based on the provided text.
User: Text: {text}
Assistant: I've read this text.
User: What describes {entity_type} in the text?
Assistant:"""
    
    def __init__(self, model, tokenizer, max_new_tokens: int = 256):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        
        # Set pad token if not set
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def extract(self, text: str, entity_type: str) -> List[str]:
        """
        Extract entities of a specific type from text.
        
        Args:
            text: The input text to extract entities from
            entity_type: The type of entity to extract (e.g., "person", "organization")
            
        Returns:
            List of extracted entity strings
        """
        # Format the prompt
        prompt = self.PROMPT_TEMPLATE.format(
            text=text.strip(),
            entity_type=entity_type.lower()
        )
        
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(self.model.device)
        
        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode only the new tokens
        response = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()
        
        # Parse the response
        return self._parse_response(response)
    
    def _parse_response(self, response: str) -> List[str]:
        """
        Parse the model response to extract entity list.
        UniNER typically responds with entities in a list format or as JSON.
        """
        # Handle "none" or empty responses
        if not response or response.lower() in ['none', 'none.', 'n/a', '[]']:
            return []
        
        # Try JSON parsing first
        try:
            entities = json.loads(response)
            if isinstance(entities, list):
                return [str(e).strip() for e in entities if e]
        except json.JSONDecodeError:
            pass
        
        # Try parsing as comma-separated or newline-separated list
        # Remove common prefixes/suffixes
        response = re.sub(r'^(The |In this text, |Based on the text, )', '', response, flags=re.IGNORECASE)
        response = response.rstrip('.')
        
        # Split by common delimiters
        if ',' in response:
            entities = [e.strip().strip('"').strip("'") for e in response.split(',')]
        elif '\n' in response:
            entities = [e.strip().strip('-').strip('•').strip() for e in response.split('\n')]
        else:
            # Single entity
            entities = [response.strip()]
        
        # Filter empty strings
        return [e for e in entities if e and e.lower() != 'none']
    
    def extract_multiple(self, text: str, entity_types: List[str]) -> Dict[str, List[str]]:
        """
        Extract multiple entity types from text.
        
        Args:
            text: The input text
            entity_types: List of entity types to extract
            
        Returns:
            Dictionary mapping entity types to lists of entities
        """
        results = {}
        for entity_type in entity_types:
            results[entity_type] = self.extract(text, entity_type)
        return results

# Initialize the extractor
extractor = UniNERExtractor(model, tokenizer)
print("✓ UniNER Extractor initialized!")

## 5. Basic Entity Extraction

Let's test UniNER on various entity types.

In [ ]:
# Test text
text = """
Apple Inc. announced that Tim Cook will present the new iPhone 15 at their 
headquarters in Cupertino, California on September 12, 2023. The event, 
called "Wonderlust", will also feature updates to the Apple Watch and AirPods. 
Analysts from Goldman Sachs predict the stock price will reach $200 by year end.
"""

print("="*70)
print("INPUT TEXT:")
print("="*70)
print(text.strip())
print("="*70)

# Extract different entity types
entity_types = ["organization", "person", "location", "product", "date", "event"]

print("\nEXTRACTED ENTITIES:")
print("-"*70)

for etype in entity_types:
    entities = extractor.extract(text, etype)
    print(f"\n{etype.upper()}:")
    if entities:
        for e in entities:
            print(f"  • {e}")
    else:
        print("  (none found)")

## 6. Domain-Specific Entity Extraction

### 6.1 Medical/Scientific Domain

UniNER excels at extracting specialized entities that traditional models can't handle.

In [ ]:
# Medical text
medical_text = """
The patient was prescribed Metformin 500mg twice daily for Type 2 Diabetes Mellitus. 
Laboratory results showed elevated HbA1c levels at 8.2%. The physician recommended 
monitoring blood glucose levels and scheduled a follow-up with Dr. Sarah Johnson 
at Mayo Clinic. Additional tests for thyroid function (TSH, T3, T4) were ordered.
"""

print("MEDICAL TEXT EXTRACTION")
print("="*70)
print(medical_text.strip())
print("="*70)

medical_entities = [
    "medication",
    "disease",
    "medical test",
    "dosage",
    "medical institution",
    "doctor"
]

print("\nExtracted Medical Entities:")
for etype in medical_entities:
    entities = extractor.extract(medical_text, etype)
    print(f"\n{etype.upper()}:")
    for e in entities:
        print(f"  • {e}")

In [ ]:
# Legal text
legal_text = """
In the case of Smith v. Johnson Corp (2023), the Ninth Circuit Court of Appeals 
ruled that Section 230 of the Communications Decency Act does not provide immunity 
for algorithmic amplification of harmful content. Judge Martinez, writing for the 
majority, cited precedents from Gonzalez v. Google and Twitter v. Taamneh. The 
defendant's counsel, Attorney Rebecca Chen from Davis Polk & Wardwell, filed a 
motion for certiorari to the Supreme Court.
"""

print("\nLEGAL TEXT EXTRACTION")
print("="*70)
print(legal_text.strip())
print("="*70)

legal_entities = [
    "court case",
    "legal statute",
    "court",
    "judge",
    "law firm",
    "legal concept"
]

print("\nExtracted Legal Entities:")
for etype in legal_entities:
    entities = extractor.extract(legal_text, etype)
    print(f"\n{etype.upper()}:")
    for e in entities:
        print(f"  • {e}")

### 6.2 Technical/Code Domain

In [ ]:
# Technical text
tech_text = """
The new version of TensorFlow 2.14 introduces native support for JAX interoperability. 
Developers can now use the tf.function decorator with jax.numpy operations. The update 
also includes a new KerasCV library for computer vision tasks, with pre-trained 
ResNet-50 and EfficientNet-B7 models. Installation requires Python 3.9+ and CUDA 11.8 
for GPU acceleration. The API follows the PEP 8 style guide.
"""

print("TECHNICAL TEXT EXTRACTION")
print("="*70)
print(tech_text.strip())
print("="*70)

tech_entities = [
    "software library",
    "programming language",
    "machine learning model",
    "version number",
    "API",
    "coding standard"
]

print("\nExtracted Technical Entities:")
for etype in tech_entities:
    entities = extractor.extract(tech_text, etype)
    print(f"\n{etype.upper()}:")
    for e in entities:
        print(f"  • {e}")

## 7. Comparing UniNER with GLiNER

Let's compare the two leading zero-shot NER approaches on the same texts.

In [ ]:
from gliner import GLiNER

# Load GLiNER
gliner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
print("✓ GLiNER loaded!")

In [ ]:
def compare_extractors(text: str, entity_types: List[str]):
    """
    Compare UniNER and GLiNER on the same text.
    """
    print("="*80)
    print("COMPARISON: UniNER vs GLiNER")
    print("="*80)
    print(f"\nText: {text[:200]}..." if len(text) > 200 else f"\nText: {text}")
    print("-"*80)
    
    for etype in entity_types:
        print(f"\n>>> Entity Type: {etype.upper()}")
        
        # UniNER extraction
        uniner_entities = extractor.extract(text, etype)
        
        # GLiNER extraction
        gliner_results = gliner_model.predict_entities(text, [etype], threshold=0.5)
        gliner_entities = [r['text'] for r in gliner_results]
        
        print(f"  UniNER:  {uniner_entities if uniner_entities else '(none)'}")
        print(f"  GLiNER:  {gliner_entities if gliner_entities else '(none)'}")

# Compare on general text
comparison_text = """
Elon Musk, CEO of Tesla and SpaceX, announced a partnership with NVIDIA to develop 
next-generation autonomous driving systems using the H100 GPU. The deal, worth an 
estimated $5 billion, will be finalized at their Austin headquarters in Q2 2024.
"""

compare_extractors(comparison_text, ["person", "company", "product", "money", "date"])

In [ ]:
# Compare on challenging domain-specific text
specialized_text = """
The CRISPR-Cas9 gene editing technique developed by Jennifer Doudna and 
Emmanuelle Charpentier enables precise modifications to DNA sequences. 
Their work on the PAM sequence recognition mechanism was published in Science.
"""

compare_extractors(
    specialized_text, 
    ["scientist", "scientific technique", "biological molecule", "scientific journal"]
)

### 7.1 Comparison Summary

| Aspect | UniNER | GLiNER |
|--------|--------|--------|
| **Model Size** | 7B parameters | 300M parameters |
| **Speed** | Slower (autoregressive) | Faster (encoder-based) |
| **Memory** | ~4-14GB | ~1-2GB |
| **Zero-shot Quality** | Better on rare entities | Good on common entities |
| **Output Format** | Text list | Spans with positions |
| **Batch Processing** | Limited | Efficient |
| **Fine-tuning** | Complex (full LLM) | Simple (span classifier) |

## 8. Advanced Features

### 8.1 Negative Examples and Entity Disambiguation

In [ ]:
# Test disambiguation between similar entity types
ambiguous_text = """
Washington led his troops across the Delaware River to reach Washington, D.C.
The Washington Post reported that Washington state experienced flooding.
"""

print("ENTITY DISAMBIGUATION TEST")
print("="*70)
print(ambiguous_text.strip())
print("="*70)

disambiguation_types = [
    "person",
    "city",
    "U.S. state",
    "newspaper",
    "river"
]

print("\nDisambiguated Entities:")
for etype in disambiguation_types:
    entities = extractor.extract(ambiguous_text, etype)
    print(f"  {etype}: {entities}")

### 8.2 Nested Entity Extraction

UniNER can handle nested entities where one entity is contained within another.

In [ ]:
# Nested entity example
nested_text = """
The Bank of America Tower in New York City was designed by Cook+Fox Architects.
"""

print("NESTED ENTITY EXTRACTION")
print("="*70)
print(nested_text.strip())
print("="*70)

nested_types = [
    "building",
    "organization",  # Bank of America (within building name)
    "city",
    "architecture firm"
]

print("\nExtracted (including nested):")
for etype in nested_types:
    entities = extractor.extract(nested_text, etype)
    print(f"  {etype}: {entities}")

### 8.3 Batch Processing for Efficiency

In [ ]:
class BatchUniNERExtractor:
    """Batch processing for UniNER to improve throughput."""
    
    PROMPT_TEMPLATE = """A virtual assistant answers questions from a user based on the provided text.
User: Text: {text}
Assistant: I've read this text.
User: What describes {entity_type} in the text?
Assistant:"""
    
    def __init__(self, model, tokenizer, batch_size: int = 4):
        self.model = model
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    
    def extract_batch(
        self, 
        texts: List[str], 
        entity_type: str
    ) -> List[List[str]]:
        """
        Extract entities from multiple texts in batches.
        
        Args:
            texts: List of input texts
            entity_type: Entity type to extract
            
        Returns:
            List of entity lists for each text
        """
        all_results = []
        
        for i in range(0, len(texts), self.batch_size):
            batch_texts = texts[i:i + self.batch_size]
            
            # Prepare prompts
            prompts = [
                self.PROMPT_TEMPLATE.format(
                    text=text.strip(),
                    entity_type=entity_type.lower()
                )
                for text in batch_texts
            ]
            
            # Tokenize with padding
            inputs = self.tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048
            ).to(self.model.device)
            
            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=128,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=self.tokenizer.pad_token_id
                )
            
            # Decode and parse each output
            for j, output in enumerate(outputs):
                response = self.tokenizer.decode(
                    output[inputs['input_ids'].shape[1]:],
                    skip_special_tokens=True
                ).strip()
                
                entities = self._parse_response(response)
                all_results.append(entities)
        
        return all_results
    
    def _parse_response(self, response: str) -> List[str]:
        """Parse model response to extract entities."""
        if not response or response.lower() in ['none', 'none.', 'n/a', '[]']:
            return []
        
        try:
            entities = json.loads(response)
            if isinstance(entities, list):
                return [str(e).strip() for e in entities if e]
        except json.JSONDecodeError:
            pass
        
        response = response.rstrip('.')
        if ',' in response:
            entities = [e.strip().strip('"').strip("'") for e in response.split(',')]
        else:
            entities = [response.strip()]
        
        return [e for e in entities if e and e.lower() != 'none']

# Test batch processing
batch_extractor = BatchUniNERExtractor(model, tokenizer, batch_size=2)

test_texts = [
    "Microsoft announced a partnership with OpenAI.",
    "Google's CEO Sundar Pichai spoke at the conference.",
    "Amazon is expanding its AWS services in Europe.",
    "Meta released a new version of Llama."
]

print("Batch extraction of 'organization' entities:")
results = batch_extractor.extract_batch(test_texts, "organization")
for text, entities in zip(test_texts, results):
    print(f"  Text: {text}")
    print(f"  Entities: {entities}\n")

## 9. Performance Benchmarks

UniNER achieves strong zero-shot performance across standard NER benchmarks.

In [ ]:
import pandas as pd

# Benchmark results from the UniNER paper
benchmark_data = {
    "Model": [
        "ChatGPT (few-shot)",
        "InstructUIE-11B",
        "GLiNER-L",
        "UniNER-7B-type",
        "UniNER-7B-all",
        "BERT-base (supervised)"
    ],
    "CoNLL03 (F1)": [54.0, 79.1, 86.5, 89.2, 90.0, 91.2],
    "OntoNotes (F1)": [35.6, 54.8, 62.3, 71.4, 73.8, 88.5],
    "WNUT17 (F1)": [33.5, 41.2, 52.1, 54.3, 56.8, 47.3],
    "CrossNER (Avg)": [44.2, 58.9, 62.4, 68.2, 71.5, "N/A"],
    "Training": [
        "Zero-shot",
        "Instruction-tuned",
        "Zero-shot",
        "Type-distilled",
        "All-distilled",
        "Supervised"
    ]
}

df = pd.DataFrame(benchmark_data)
print("\nZERO-SHOT NER BENCHMARK RESULTS")
print("="*80)
print(df.to_string(index=False))
print("\nNote: UniNER-7B-all achieves near-supervised performance in zero-shot setting!")
print("Source: Zhou et al., 2023 - UniversalNER Paper")

## 10. Building a Production Pipeline

### 10.1 Complete UniNER Service

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Optional
import time

@dataclass
class EntityResult:
    """Structured result for extracted entities."""
    text: str
    entity_type: str
    confidence: float = 1.0  # UniNER doesn't provide confidence scores

class UniNERService:
    """
    Production-ready service for UniNER entity extraction.
    
    Features:
    - Caching for repeated queries
    - Configurable entity types
    - Timing statistics
    - Error handling
    """
    
    DEFAULT_ENTITY_TYPES = [
        "person", "organization", "location", "date", 
        "money", "product", "event"
    ]
    
    def __init__(
        self, 
        model, 
        tokenizer,
        entity_types: Optional[List[str]] = None,
        use_cache: bool = True
    ):
        self.extractor = UniNERExtractor(model, tokenizer)
        self.entity_types = entity_types or self.DEFAULT_ENTITY_TYPES
        self.use_cache = use_cache
        self._cache = {}
        self._stats = {"requests": 0, "cache_hits": 0, "total_time": 0}
    
    def extract(
        self, 
        text: str, 
        entity_types: Optional[List[str]] = None
    ) -> Dict[str, List[EntityResult]]:
        """
        Extract all configured entity types from text.
        
        Args:
            text: Input text
            entity_types: Override default entity types
            
        Returns:
            Dictionary mapping entity types to extracted entities
        """
        start_time = time.time()
        self._stats["requests"] += 1
        
        types_to_extract = entity_types or self.entity_types
        results = {}
        
        for etype in types_to_extract:
            cache_key = f"{hash(text)}:{etype}"
            
            if self.use_cache and cache_key in self._cache:
                self._stats["cache_hits"] += 1
                results[etype] = self._cache[cache_key]
            else:
                try:
                    entities = self.extractor.extract(text, etype)
                    entity_results = [
                        EntityResult(text=e, entity_type=etype)
                        for e in entities
                    ]
                    results[etype] = entity_results
                    
                    if self.use_cache:
                        self._cache[cache_key] = entity_results
                except Exception as e:
                    print(f"Warning: Error extracting {etype}: {e}")
                    results[etype] = []
        
        self._stats["total_time"] += time.time() - start_time
        return results
    
    def get_stats(self) -> Dict:
        """Get service statistics."""
        return {
            **self._stats,
            "cache_size": len(self._cache),
            "avg_time": self._stats["total_time"] / max(1, self._stats["requests"])
        }
    
    def clear_cache(self):
        """Clear the cache."""
        self._cache.clear()

# Initialize service
service = UniNERService(
    model, 
    tokenizer,
    entity_types=["person", "organization", "location", "product"]
)

# Test the service
test_text = """
Satya Nadella announced Microsoft's acquisition of Activision Blizzard for $69 billion.
The deal includes popular games like Call of Duty and World of Warcraft.
"""

print("UniNER Service Test:")
print("="*70)
results = service.extract(test_text)

for etype, entities in results.items():
    print(f"\n{etype.upper()}:")
    for entity in entities:
        print(f"  • {entity.text}")

print(f"\n\nService Stats: {service.get_stats()}")

## 11. Limitations and Best Practices

### 11.1 Known Limitations

1. **Memory Requirements**: Even quantized, UniNER needs ~4GB VRAM
2. **Speed**: Autoregressive generation is slower than encoder models
3. **No Span Positions**: Unlike GLiNER, UniNER doesn't provide character offsets
4. **Hallucination Risk**: May generate entities not in the source text
5. **Inconsistent Output Format**: Response parsing can be challenging

### 11.2 Best Practices

In [ ]:
# Best Practice 1: Validate extracted entities exist in source text
def validate_entities(text: str, entities: List[str]) -> List[str]:
    """Filter entities to only those appearing in the source text."""
    text_lower = text.lower()
    return [e for e in entities if e.lower() in text_lower]

# Example
text = "Apple released the iPhone 15 Pro Max."
raw_entities = ["Apple", "iPhone 15 Pro Max", "Samsung"]  # Samsung is hallucinated
validated = validate_entities(text, raw_entities)
print(f"Raw: {raw_entities}")
print(f"Validated: {validated}")

In [ ]:
# Best Practice 2: Use specific entity type descriptions
# Instead of: "organization"
# Use: "technology company" or "financial institution"

finance_text = """
Goldman Sachs analysts predict that the Federal Reserve will cut rates.
Meanwhile, startups like Stripe and Plaid are disrupting traditional banking.
"""

# Generic vs specific entity types
generic_orgs = extractor.extract(finance_text, "organization")
investment_banks = extractor.extract(finance_text, "investment bank")
fintech = extractor.extract(finance_text, "fintech company")
central_banks = extractor.extract(finance_text, "central bank")

print("Generic 'organization':     ", generic_orgs)
print("Specific 'investment bank': ", investment_banks)
print("Specific 'fintech company': ", fintech)
print("Specific 'central bank':    ", central_banks)

In [ ]:
# Best Practice 3: Post-process for deduplication
def deduplicate_entities(entities: List[str]) -> List[str]:
    """Remove duplicate entities (case-insensitive)."""
    seen = set()
    unique = []
    for e in entities:
        lower = e.lower()
        if lower not in seen:
            seen.add(lower)
            unique.append(e)
    return unique

# Example with repeated entities
entities = ["Apple Inc.", "apple inc.", "Apple", "Microsoft", "microsoft"]
print(f"Original: {entities}")
print(f"Deduplicated: {deduplicate_entities(entities)}")

## 12. Summary and When to Use UniNER

### Key Takeaways

1. **UniNER is best for**:
   - Zero-shot extraction of rare/domain-specific entity types
   - Applications where quality matters more than speed
   - Scenarios with many diverse entity types
   - Research and prototyping

2. **UniNER is NOT ideal for**:
   - High-throughput production systems
   - Resource-constrained environments
   - When character-level positions are needed
   - Real-time applications

### Decision Guide

```
Need zero-shot NER?
├── Yes
│   ├── Need span positions?
│   │   ├── Yes → GLiNER
│   │   └── No
│   │       ├── Have GPU with 4GB+ VRAM?
│   │       │   ├── Yes → UniNER (better quality)
│   │       │   └── No → GLiNER (lower memory)
│   │       └── Need very rare entity types?
│   │           ├── Yes → UniNER
│   │           └── No → GLiNER (faster)
│   └── High throughput needed?
│       ├── Yes → GLiNER
│       └── No → UniNER
└── No (have training data)
    └── Fine-tune BERT/SpanMarker
```

### Further Reading

- [UniversalNER Paper](https://arxiv.org/abs/2308.03279)
- [GLiNER Paper](https://arxiv.org/abs/2311.08526)
- [Zero-shot NER Survey](https://arxiv.org/abs/2303.03886)
- [Llama Fine-tuning Guide](https://huggingface.co/docs/transformers/main/en/llama2)

## 13. Exercises

### Exercise 1: Custom Domain Extraction
Build a UniNER pipeline for extracting entities from a domain of your choice (e.g., recipes, sports, music).

### Exercise 2: Hybrid System
Create a system that uses GLiNER for common entities (PER, ORG, LOC) and UniNER for specialized entities.

### Exercise 3: Evaluation Pipeline
Implement evaluation on a subset of CoNLL-2003 using seqeval metrics.

### Exercise 4: Entity Linking
Extend the pipeline to link extracted entities to Wikipedia/Wikidata.

In [ ]:
# Exercise 1 Starter: Recipe Domain
recipe_text = """
Preheat the oven to 375°F. In a large bowl, combine 2 cups of all-purpose flour, 
1 teaspoon of baking powder, and a pinch of salt. Add 1/2 cup of softened butter 
and mix until crumbly. The preparation time is about 15 minutes.
"""

recipe_entities = [
    "cooking equipment",
    "ingredient",
    "measurement",
    "temperature",
    "cooking action",
    "time duration"
]

print("Recipe Entity Extraction:")
print("="*70)
for etype in recipe_entities:
    entities = extractor.extract(recipe_text, etype)
    print(f"{etype}: {entities}")

In [ ]:
# Cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
print("\n" + "="*70)
print("Lesson 10 Complete: UniversalNER / UniNER")
print("="*70)
print("""
You've learned:
✓ How UniNER uses targeted distillation from ChatGPT
✓ The instruction format for open-domain NER
✓ How to use 4-bit quantization for efficient inference
✓ Domain-specific entity extraction
✓ Comparison with GLiNER
✓ Best practices and limitations
✓ Building production pipelines

Next: Lesson 11 - Flair NER (Contextual String Embeddings)
""")